# Hyperparameter Sweeper (TensorFlow + KerasTuner)
Explores learning-rate, width, dropout, and regularization combinations for the MLP using the engineered features. Run after `feature_engineering_playbook.ipynb`.

## Strategy
- Use [`keras-tuner`](https://keras.io/keras_tuner/) RandomSearch for up to N trials (increase when using GPU/Colab).
- Targets ROC-AUC on the validation set.
- Outputs best hyperparameters so you can copy them into the main MLP notebook.

### Search space
| Hyperparameter | Values |
| --- | --- |
| Hidden units per layer | `[128, 64]`, `[96, 48]`, `[64, 32]` |
| Dropout rate | 0.2 – 0.5 |
| L2 penalty | 1e-5 – 1e-3 (log scale) |
| Learning rate | 5e-4 – 3e-3 |
| Batch size | 32, 64, 96 |


In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import tensorflow as tf
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'data').exists():
    if REPO_ROOT == REPO_ROOT.parent:
        raise RuntimeError('Could not find project root containing data/')
    REPO_ROOT = REPO_ROOT.parent

print(f'Using repo root: {REPO_ROOT}')
engineered_path = REPO_ROOT / 'data/processed/engineered_features.csv'
if not engineered_path.exists():
    raise FileNotFoundError('Run feature_engineering_playbook.ipynb first to create engineered_features.csv')

df = pd.read_csv(engineered_path)
TARGET = 'blueWins'
FEATURES = [col for col in df.columns if col != TARGET]
X_train, X_val, y_train, y_val = train_test_split(
    df[FEATURES], df[TARGET], test_size=0.2, stratify=df[TARGET], random_state=123
)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)


## Define search space

In [ ]:
def build_model(hp: kt.HyperParameters):
    hidden_config = hp.Choice('hidden_config', values=[(128, 64), (96, 48), (64, 32)])
    dropout_rate = hp.Float('dropout_rate', 0.2, 0.5, step=0.05)
    l2_value = hp.Float('l2', 1e-5, 1e-3, sampling='log')
    learning_rate = hp.Float('learning_rate', 5e-4, 3e-3, sampling='log')

    model = tf.keras.Sequential(name='mlp_tuned')
    model.add(tf.keras.layers.Input(shape=(X_train_scaled.shape[1],)))
    for units in hidden_config:
        model.add(tf.keras.layers.Dense(
            units,
            activation='relu',
            kernel_regularizer=tf.keras.regularizers.l2(l2_value),
        ))
        model.add(tf.keras.layers.Dropout(dropout_rate))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=[tf.keras.metrics.AUC(name='roc_auc')],
    )
    return model


## Launch Random Search
Increase `max_trials`/`executions_per_trial` when you have more compute.

In [ ]:
tuner = kt.RandomSearch(
    build_model,
    objective=kt.Objective('val_roc_auc', direction='max'),
    max_trials=15,
    executions_per_trial=1,
    overwrite=True,
    directory=(REPO_ROOT / 'tuner_logs').as_posix(),
    project_name='mlp_sweeps',
)

tuner.search(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=100,
    batch_size=kt.Int('batch_size', 32, 96, step=32),
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_roc_auc', patience=10, mode='max', restore_best_weights=True)],
    verbose=1,
)


## Inspect best trial

In [ ]:
best_hp = tuner.get_best_hyperparameters(1)[0]
print('Best hyperparameters:')
for param in ['hidden_config', 'dropout_rate', 'l2', 'learning_rate', 'batch_size']:
    print(f"  {param}: {best_hp.get(param)}")

print('
Rebuild + evaluate best model:')
best_model = tuner.hypermodel.build(best_hp)
history = best_model.fit(
    X_train_scaled,
    y_train,
    validation_data=(X_val_scaled, y_val),
    epochs=150,
    batch_size=best_hp.get('batch_size'),
    callbacks=[tf.keras.callbacks.EarlyStopping(monitor='val_roc_auc', patience=12, mode='max', restore_best_weights=True)],
    verbose=0,
)
val_probs = best_model.predict(X_val_scaled).flatten()
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
val_preds = (val_probs >= 0.5).astype(int)
print('Validation Accuracy:', accuracy_score(y_val, val_preds))
print('Validation F1:', f1_score(y_val, val_preds))
print('Validation ROC-AUC:', roc_auc_score(y_val, val_probs))


### Next steps
- Copy `best_hp` values into `mlp_tensorflow.ipynb` or your training scripts.
- Increase `max_trials`/`executions_per_trial` with GPU time for deeper searches.
- Swap `build_model` to target other architectures (e.g., wide & deep) while reusing this notebook structure.